# Part 8 · The tool layer: GitHub, MCP and the gateway

**The story:** a developer wires GitHub's MCP server to an agent and it works, so
they wire up two more. Now every tool definition from every server is in the model's
context on every turn, the model has started reaching for the wrong one, and nobody
can say which of those tools the agent was ever supposed to call. This part is about
the layer that fixes that, which is not in the agent's code.

We front **GitHub's hosted MCP server** with agentgateway, publish it in
**AgentRegistry** as an approved tool server, scaffold an agent against it with
`arctl`, run it on **kagent**, and then work through the four `toolMode` settings on
one field, measuring each. It finishes with the agent being told to merge a pull
request and not being able to.

Runs on **`mesh1`** and needs the Part 4 platform (`demo-scripts/agentregistry/setup-mesh1.sh`),
plus a GitHub PAT. Read access is enough: the write tools are denied at the gateway
in §9 rather than withheld from the token, which is the point.

### What is actually being measured

Every number in this notebook came off this cluster. The four settings are not four
speeds of the same thing, they trade different resources, and the interesting result
is that the cheapest context got the answer wrong.

## Setup

One standup, then the Connect cell. `GITHUB_PAT` is read from the environment (the
suite's secrets file exports `GITHUB_PORTLAB_TOKEN`, which is also accepted).

In [ ]:
# from the folder that holds this notebook (the one containing demo-scripts/)
set -a; . "${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}" 2>/dev/null; set +a
source demo-scripts/agentregistry/scripts/connect.sh >/dev/null 2>&1
./demo-scripts/prtriage/scripts/setup.sh

### Connect

In [ ]:
# walk up to the suite root so this works from wherever an earlier cell left us
LAB=$PWD
until [ -d "$LAB/demo-scripts/prtriage" ]; do
  [ "$LAB" = / ] && { echo "✗ run this from the demo suite folder (the one with demo-scripts/)"; break; }
  LAB=$(dirname "$LAB")
done
cd "$LAB"
set -a; . "${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}" 2>/dev/null; set +a
source demo-scripts/agentregistry/scripts/connect.sh
export PART8=demo-scripts/prtriage
export LB=$(kubectl --context kind-mesh1 -n agentgateway-system get gateway ar-ingress -o jsonpath='{.status.addresses[0].value}')
export MCP="http://github-mcp.${LB}.sslip.io/"
export ASK="demo-scripts/agentregistry/scripts/ask.sh"
echo
printf "  %-20s %s\n" "MCP endpoint:"     "$MCP"
printf "  %-20s %s\n" "AgentRegistry UI:" "http://${AR_HOST}"
printf "  %-20s %s\n" "kagent UI:"        "http://${KAGENT_UI_HOST}  (admin-user / password)"

### A tiny MCP client

`mcp.sh` is a few lines of curl: initialize, keep the session id, then send one
JSON-RPC call. Everything in the first half of this notebook goes through it, so you
can see the wire rather than a framework's idea of it.

In [ ]:
cat > /tmp/mcp.sh <<'SH'
#!/usr/bin/env bash
# mcp.sh <endpoint> <method> [params-json] — one MCP call, session handled.
set -euo pipefail
EP="$1"; METHOD="$2"; PARAMS="${3:-}"
H=(-H "Content-Type: application/json" -H "Accept: application/json, text/event-stream")
HDR=$(mktemp)
curl -s -m 60 -X POST "$EP" "${H[@]}" -D "$HDR" -o /dev/null \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-06-18","capabilities":{},"clientInfo":{"name":"demo8","version":"1"}}}'
SID=$(grep -i '^mcp-session-id:' "$HDR" | tr -d '\r' | awk '{print $2}')
BODY=$(python3 -c 'import json,sys;print(json.dumps({"jsonrpc":"2.0","id":2,"method":sys.argv[1],"params":json.loads(sys.argv[2] or "{}")}))' "$METHOD" "$PARAMS")
curl -s -m 180 -X POST "$EP" "${H[@]}" ${SID:+-H "Mcp-Session-Id: $SID"} -d "$BODY" \
  | sed 's/^data: //' | grep -v '^event:' | grep -v '^$'
SH
chmod +x /tmp/mcp.sh; echo "wrote /tmp/mcp.sh"

## 1. What connecting one MCP server actually costs

GitHub's hosted MCP server is one server, and this is what it puts in the model's
context on every turn. Count the tools, weigh the schema, and count how many of
them can change your repository.

In [ ]:
/tmp/mcp.sh "$MCP" tools/list > /tmp/tools-standard.json
python3 - /tmp/tools-standard.json <<'PY'
import json,sys,re
d=json.load(open(sys.argv[1])); t=d["result"]["tools"]
print("tools:", len(t))
print("tools/list payload:", len(json.dumps(d)), "bytes")
WRITE=("create_","update_","delete_","merge_","push_","add_","fork_","_write","request_copilot_review")
w=[x["name"] for x in t if any(k in x["name"] for k in WRITE)]
print("of which can write:", len(w))
print()
print("the write tools:"); print("  " + ", ".join(sorted(w)))
PY

Now the number that matters, from Anthropic's own token counter rather than a
divide-by-four estimate. This is the floor on every turn, before the user has typed
anything.

In [ ]:
python3 - /tmp/tools-standard.json <<'PY' > /tmp/count.json
import json,sys
d=json.load(open(sys.argv[1]))
tools=[{"name":t["name"],"description":t.get("description",""),
        "input_schema":t.get("inputSchema",{"type":"object"})} for t in d["result"]["tools"]]
print(json.dumps({"model":"claude-sonnet-4-5","tools":tools,
                  "messages":[{"role":"user","content":"hi"}]}))
PY
curl -s https://api.anthropic.com/v1/messages/count_tokens \
  -H "x-api-key: $ANTHROPIC_API_KEY" -H "anthropic-version: 2023-06-01" \
  -H "content-type: application/json" -d @/tmp/count.json

Two things to take from that, and neither is fixable in your agent code:

1. Around **14,500 tokens** of tool schema on every turn, from a single server.
2. **17 of the 44 tools write.** `delete_file`, `merge_pull_request`, `push_files`.
   A PAT scope cannot express "this one agent may only read pull requests", because
   a scope is coarse and the token is the same one for everyone who uses it.

## 2. The gateway holds the credential, so the agent never does

`setup.sh` already applied this. It is worth reading, because two fields carry the
whole idea: `protocol: StreamableHTTP` reaches GitHub's hosted server, and
`policies.auth.secretRef` injects the PAT upstream from a Secret the gateway reads.

The agent gets a plain HTTP URL with no credential in it, and it cannot leak a token
it was never given.

In [ ]:
# just the backend document (the HTTPRoute after it is ordinary Gateway API)
sed -n '/^apiVersion/,$p' $PART8/yaml/10-github-backend.yaml | sed '/^---$/,$d'

Proof, rather than assertion: this call carries **no `Authorization` header at all**,
and GitHub answers with real data.

In [ ]:
echo "  == what the client sends: no credential =="
echo "  POST $MCP   (Content-Type + Accept only)"
echo
/tmp/mcp.sh "$MCP" tools/call \
  '{"name":"list_pull_requests","arguments":{"owner":"kagent-dev","repo":"kagent","state":"open","perPage":3,"fields":["number","title"]}}' \
  | python3 -c "
import json,sys
d=json.load(sys.stdin)
rows=json.loads(d['result']['content'][0]['text'])
for r in rows: print('  #%s  %s' % (r['number'], r['title'][:64]))
"

## 3. The catalogue entry points at the gateway

This is the governance move, and it is one line of YAML. The approved MCP server in
the registry resolves to the **agentgateway route**, not to `api.githubcopilot.com`.
A developer picking GitHub out of the catalogue gets GitHub through the policy
enforcement point, and there is no catalogue entry that means "GitHub, but skip the
gateway".

In [ ]:
arctl get mcpserver github-mcp -o yaml | sed -n '/^spec:/,$p'

## 4. Scaffold the agent from approved parts

`arctl init` writes a complete, runnable ADK + Python project. The flags are the
governance story: `--mcp github-mcp@latest` wires it to the approved server above,
and the model is pinned.

The scaffold ships two sample tools (`roll_die`, `check_prime`). We replace them
with **one** local tool, `today`, and that one exists for a reason you will see in
§7: the gateway's code sandbox has no clock, so the agent has to supply the date.
Everything the agent can do to GitHub arrives through the catalogue.

In [ ]:
# scaffold it, then swap the sample tools out for ours. FORCE=1 starts from scratch.
FORCE=1 $PART8/scripts/scaffold-agent.sh
echo
echo "  == what it was wired to =="
grep -A4 mcpServers prtriage/agent.yaml

`roll_die` and `check_prime` were the scaffold's samples. `today` is what replaces
them, and it is the only local tool the agent has, because the gateway's code sandbox
has no clock. Everything it can do to GitHub arrives through the catalogue.

In [ ]:
sed -n '/^def today/,/^    return/p' prtriage/prtriage/agent.py
echo
echo "  == and the comment above the tool list =="
sed -n '/^# Everything this agent/,/^mcp_tools/p' prtriage/prtriage/agent.py

### The approved skill is where the hard-won details live

A **skill** in the catalogue is approved, versioned guidance. This one is the
platform team's write-up of how to drive this gateway correctly, and every rule in it
was put there because a run failed without it: the sandbox has no `Date`, the
parameter is `pullNumber` and not `pull_number`, `get_check_runs` returns an object
while `get_reviews` returns an array, a program gets at most 20 upstream calls.

That is the argument for a registry in one file. The team learns this once.

In [ ]:
arctl get skill release-report -o yaml | sed -n '/^spec:/,$p'
echo
echo "  == the rules the platform team recorded =="
sed -n '/## Rules for the program/,/## House format/p' $PART8/skill/release-report/SKILL.md

The skill body is baked into the agent's `prompts.json` at build time, and
`build_instruction()` in the scaffold makes it the system instruction. `rebuild-agent.sh`
does that, builds, pushes and restarts.

It also works around a trap that will cost anyone an hour: the image is
`localhost:5001/prtriage:latest` and kagent runs it `IfNotPresent`, so pushing a new
image on the same tag and restarting quietly reuses the node's cached copy. You then
debug a change that was never deployed. The script drops the cached image from every
node first, and checks the running pod really has the current skill.

In [ ]:
$PART8/scripts/rebuild-agent.sh

## 5. Deploy it on kagent and watch Standard mode work

`Standard` is the default `toolMode`: one MCP tool per GitHub operation. Ask for a
report on nine pull requests and watch what the model has to do.

In [ ]:
arctl apply -f $PART8/yaml/40-deploy-kagent.yaml
kubectl --context kind-mesh1 -n kagent rollout status deploy/prtriage --timeout=240s
kubectl --context kind-mesh1 -n kagent get pods | grep -E 'NAME|prtriage'

In [ ]:
kubectl --context kind-mesh1 -n agentgateway-system patch enterpriseagentgatewaybackend github-mcp \
  --type=merge -p '{"spec":{"entMcp":{"toolMode":"Standard"}}}'
# the agent lists tools once at startup, so it has to re-list to see a mode change
kubectl --context kind-mesh1 -n kagent rollout restart deploy/prtriage
kubectl --context kind-mesh1 -n kagent rollout status deploy/prtriage --timeout=240s
sleep 5
AGENT_PREFIX=prtriage $ASK "Give me the release report for kagent-dev/kagent, the 9 most recently opened pull requests."

Read that trace rather than the answer. Twenty tool calls, each one a round trip that
re-sends the whole conversation, and look at what came back through the model on the
way: complete Copilot review bodies, file-by-file tables, HTML entities. Around
**87,000 bytes** of intermediate JSON passed through the context window to produce
eleven lines of report.

Nothing here is misconfigured. This is what one MCP server and a wide question cost.

## 6. Search mode: the catalogue becomes searchable

`toolMode: Search` replaces the 44 tools with two, `get_tool` and `invoke_tool`. The
tool **names** stay in `get_tool`'s description; the **schemas** are fetched on demand.

In [ ]:
kubectl --context kind-mesh1 -n agentgateway-system patch enterpriseagentgatewaybackend github-mcp \
  --type=merge -p '{"spec":{"entMcp":{"toolMode":"Search"}}}'
sleep 8
/tmp/mcp.sh "$MCP" tools/list > /tmp/tools-search.json
python3 -c "
import json
d=json.load(open('/tmp/tools-search.json')); t=d['result']['tools']
print('tools now:',len(t),'->',', '.join(x['name'] for x in t))
print('tools/list payload:',len(json.dumps(d)),'bytes')
"

In [ ]:
python3 - /tmp/tools-search.json <<'PY' > /tmp/count.json
import json,sys
d=json.load(open(sys.argv[1]))
tools=[{"name":t["name"],"description":t.get("description",""),
        "input_schema":t.get("inputSchema",{"type":"object"})} for t in d["result"]["tools"]]
print(json.dumps({"model":"claude-sonnet-4-5","tools":tools,"messages":[{"role":"user","content":"hi"}]}))
PY
curl -s https://api.anthropic.com/v1/messages/count_tokens \
  -H "x-api-key: $ANTHROPIC_API_KEY" -H "anthropic-version: 2023-06-01" \
  -H "content-type: application/json" -d @/tmp/count.json

**14,572 tokens down to 986, a 93% cut**, and that is the honest headline for Search
mode. Now run the same job on it, because the headline is not the whole story.

In [ ]:
kubectl --context kind-mesh1 -n kagent rollout restart deploy/prtriage
kubectl --context kind-mesh1 -n kagent rollout status deploy/prtriage --timeout=240s
sleep 5
AGENT_PREFIX=prtriage $ASK "Give me the release report for kagent-dev/kagent, the 9 most recently opened pull requests." \
  | tee /tmp/run-search.txt | tail -20
echo
echo "  == what that cost =="
python3 -c "
import re
L=[l for l in open('/tmp/run-search.txt') if re.match(r'^\s+\d+\.\s+\S+\(',l)]
print('  model round trips:',len(L))
print('  payload through the model:',sum(len(l.split('-> ',1)[1]) for l in L if '-> ' in l),'bytes')
"

So Search mode is **worse** for this job: measured here at 23 round trips and about
222,000 bytes through the model, against Standard's 20 and 87,000. Discovering a
tool costs a turn, invoking it costs another, and `invoke_tool` hands back the raw
response plus the schema you fetched to get there.

Search mode is the right answer when the catalogue is huge and the task touches one
or two tools. A wide aggregation is not that task. Which is the whole reason the next
setting exists.

## 7. Code mode: one program instead of twenty turns

`toolMode: Code` collapses the server to a single `run_code` tool whose description
is a generated TypeScript API over the same 44 operations. The model writes one
JavaScript program, the gateway runs it in a sandbox, makes the upstream REST calls,
and returns only what the program returns.

In [ ]:
kubectl --context kind-mesh1 -n agentgateway-system patch enterpriseagentgatewaybackend github-mcp \
  --type=merge -p '{"spec":{"entMcp":{"toolMode":"Code"}}}'
sleep 8
/tmp/mcp.sh "$MCP" tools/list > /tmp/tools-code.json
python3 -c "
import json
t=json.load(open('/tmp/tools-code.json'))['result']['tools']
print('tools now:',len(t),'->',t[0]['name'])
print('generated API in its description:',len(t[0]['description']),'chars')
print()
i=t[0]['description'].find('Available API:')
print(t[0]['description'][i:i+520])
"

### The sandbox is small on purpose

Worth probing rather than assuming, because it decides what the model is allowed to
write. The only way out of this sandbox is the generated tool functions: there is no
`fetch`, no `require`, no `process`. A program cannot phone home, it can only call
approved tools.

In [ ]:
/tmp/mcp.sh "$MCP" tools/call '{"name":"run_code","arguments":{"code":"const p={}; for (const n of [\"Date\",\"fetch\",\"Math\",\"JSON\",\"Promise\",\"Map\",\"Set\",\"console\",\"process\",\"require\",\"setTimeout\",\"RegExp\",\"BigInt\"]) { try { p[n]=typeof eval(n); } catch(e) { p[n]=\"MISSING\"; } } p"}}' \
  | python3 -c "
import json,sys
d=json.load(sys.stdin)
print(json.dumps(json.loads(d['result']['content'][0]['text'])['success'],indent=1))
"

No `Date` is why the agent keeps a local `today` tool, and no `Map` or `Set` is the
kind of thing a model reaches for by habit. Both are written down in the approved
skill, which is the only reason the run below works first time.

There is one more limit worth knowing: **a program may make at most 20 upstream tool
calls.** That is what sizes the report at nine pull requests, one list call plus two
per pull request.

In [ ]:
kubectl --context kind-mesh1 -n kagent rollout restart deploy/prtriage
kubectl --context kind-mesh1 -n kagent rollout status deploy/prtriage --timeout=240s
sleep 5
AGENT_PREFIX=prtriage $ASK "Give me the release report for kagent-dev/kagent, the 9 most recently opened pull requests." \
  | tee /tmp/run-code.txt | tail -22
echo
echo "  == what that cost =="
python3 -c "
import re
L=[l for l in open('/tmp/run-code.txt') if re.match(r'^\s+\d+\.\s+\S+\(',l)]
print('  model round trips:',len(L))
print('  payload through the model:',sum(len(l.split('-> ',1)[1]) for l in L if '-> ' in l),'bytes')
"

Two round trips. `today()`, then one program that made nineteen GitHub calls inside
the gateway and returned the finished report. The same answer as §5, from about
**10 bytes** of intermediate data through the model instead of 87,000.

## 8. CodeSearch, and the run that was confidently wrong

`CodeSearch` is the fourth setting: `get_tool` plus `run_code`, with the generated API
dropped from the description and discovered on demand. It is the cheapest of the four
on context, at **1,300 tokens**.

It is also the setting that got the answer wrong, and that is the most useful thing in
this notebook. On the first attempt, with a thinner skill, the model hit the 20-call
cap twice, used `pull_number` instead of `pullNumber`, and then wrote this:

```js
if (pr.draft) { ... }
else if (Array.isArray(prChecks.check_runs)) { ...only sets a status if something failed... }
else if (!prReviews.some(r => r.state === "APPROVED")) { status = "no approval"; }
```

The middle guard is true whenever checks came back at all, so the approval test was
never reached and every pull request with passing checks was reported **READY**. It
said eight were ready to merge. None of them had an approval.

Nothing was broken. The model had less information, retried under pressure, and wrote
a plausible bug. The fix was to record the parameter name and the classification logic
in the approved skill, which is what §4 showed. Run it now and the answer is right.

In [ ]:
kubectl --context kind-mesh1 -n agentgateway-system patch enterpriseagentgatewaybackend github-mcp \
  --type=merge -p '{"spec":{"entMcp":{"toolMode":"CodeSearch"}}}'
sleep 8
/tmp/mcp.sh "$MCP" tools/list | python3 -c "
import json,sys
t=json.load(sys.stdin)['result']['tools']
print('tools now:',len(t),'->',', '.join(x['name'] for x in t))
"
kubectl --context kind-mesh1 -n kagent rollout restart deploy/prtriage
kubectl --context kind-mesh1 -n kagent rollout status deploy/prtriage --timeout=240s
sleep 5
AGENT_PREFIX=prtriage $ASK "Give me the release report for kagent-dev/kagent, the 9 most recently opened pull requests." \
  | tee /tmp/run-cs.txt | tail -18
python3 -c "
import re
L=[l for l in open('/tmp/run-cs.txt') if re.match(r'^\s+\d+\.\s+\S+\(',l)]
print()
print('  model round trips:',len(L))
"

The lesson is not that one setting wins. It is that the cheapest context bought a
wrong answer, and five thousand more tokens of signatures bought a correct one in two
turns. Both are decisions you can only make where the tools are, which is the gateway.

## 9. Now take the write tools away

The report needs to read pull requests and nothing else. The agent has been holding
forty-five tools all along, seventeen of which write. This policy names what it
actually needs, on the MCP method name, at the gateway.

The allowlist has three entries because `run_code` is a tool too: in `Standard` mode
the agent will see the two GitHub reads, and in `Code` mode it sees only `run_code`,
whose generated API is cut down to the same two operations.

In [ ]:
cat $PART8/yaml/20-authz-readonly.yaml

In [ ]:
kubectl --context kind-mesh1 apply -f $PART8/yaml/20-authz-readonly.yaml
sleep 8
echo "  == the tool list is filtered at the gateway, not in the agent =="
kubectl --context kind-mesh1 -n agentgateway-system patch enterpriseagentgatewaybackend github-mcp \
  --type=merge -p '{"spec":{"entMcp":{"toolMode":"Standard"}}}' >/dev/null
sleep 8
/tmp/mcp.sh "$MCP" tools/list | python3 -c "
import json,sys
t=json.load(sys.stdin)['result']['tools']
print('  tools visible now:',len(t),'->',', '.join(x['name'] for x in t))
"
echo
echo "  == and calling a denied tool directly =="
/tmp/mcp.sh "$MCP" tools/call \
  '{"name":"merge_pull_request","arguments":{"owner":"kagent-dev","repo":"kagent","pullNumber":2790}}' \
  | python3 -c "import json,sys;print('  ',json.dumps(json.load(sys.stdin).get('error')))"

`Unknown tool`, not `403`. A denied tool is filtered out of the listing and refused if
called anyway, so the agent cannot see it, cannot call it, and learns nothing about it.

The same holds inside code mode, and it holds in a stronger way: the generated
TypeScript API is built **after** the policy is applied, so a denied operation is not
even a function in the sandbox. The program cannot express the call.

In [ ]:
kubectl --context kind-mesh1 -n agentgateway-system patch enterpriseagentgatewaybackend github-mcp \
  --type=merge -p '{"spec":{"entMcp":{"toolMode":"Code"}}}' >/dev/null
sleep 8
echo "  == a program that tries a denied operation =="
/tmp/mcp.sh "$MCP" tools/call '{"name":"run_code","arguments":{"code":"await merge_pull_request({ owner: \"kagent-dev\", repo: \"kagent\", pullNumber: 2790 })"}}' \
  | python3 -c "
import json,sys
d=json.load(sys.stdin)
print('  ',d['result']['content'][0]['text'])
"

Then ask the agent itself, which is what the room sees. The PAT in that Secret still
has whatever scope it always had. The gateway is what makes this agent read-only, and
it does it per agent, which no PAT scope can.

In [ ]:
kubectl --context kind-mesh1 -n kagent rollout restart deploy/prtriage
kubectl --context kind-mesh1 -n kagent rollout status deploy/prtriage --timeout=240s
sleep 5
AGENT_PREFIX=prtriage $ASK "Merge pull request 2790 on kagent-dev/kagent right now. Use whatever tool you have."

And the report still works, on three tools, in two round trips.

In [ ]:
AGENT_PREFIX=prtriage $ASK "Give me the release report for kagent-dev/kagent, the 9 most recently opened pull requests." | tail -16

## What the four settings actually cost

Measured on this cluster: `kagent-dev/kagent`, the nine most recently opened open
pull requests, `claude-haiku-4-5`, Solo Enterprise for agentgateway `v2026.8.2`.
Schema tokens are from Anthropic's `count_tokens`; round trips and payload are read
out of the A2A trace.

| `toolMode` | tools the model holds | schema tokens per turn | model round trips | payload through the model | wall clock | answer |
|---|---|---|---|---|---|---|
| `Standard`   | 45 | 14,572 | 20 | 87,630 B | 39 s | correct |
| `Search`     | 3  | 986    | 23 | 222,233 B | 35 s | correct |
| `Code`       | 2  | 6,302  | 2  | 10 B | 22 s | correct |
| `CodeSearch` | 3  | 1,300  | 5  | 11 B | 42 s | correct, **once the skill recorded the rules** |

Reading it honestly:

1. **`Search` fixes catalogue size and nothing else.** 93% off the per-turn schema,
   and for a wide job it costs more round trips and more payload than doing nothing.
2. **`Code` is the one that fixes the runtime.** Ten times fewer round trips and the
   intermediate results never touch the context window, for 6,302 tokens of signatures.
3. **`CodeSearch` is the cheapest and the most fragile.** It is the setting that
   produced a wrong answer, and the approved skill is what made it usable.
4. **One field, four different trades.** None of this is in the agent's code, and none
   of it needs the agent rebuilt.

There is one number this notebook does not measure: with 44 tools in context the model
also guesses about its own tools. Asked to count them it answered "80". Counts belong
at the gateway.

## Reset / teardown

Puts the demo back to a clean, re-runnable state. The `github-mcp-pat` Secret and the
backend stay, so `setup.sh` is only needed once.

In [ ]:
K="kubectl --context kind-mesh1"
$K -n agentgateway-system delete enterpriseagentgatewaypolicy github-readonly --ignore-not-found
$K -n agentgateway-system patch enterpriseagentgatewaybackend github-mcp \
  --type=merge -p '{"spec":{"entMcp":{"toolMode":"Standard"}}}'
$K -n kagent rollout restart deploy/prtriage
echo "✓ back to Standard mode, no policy"

Remove Part 8 entirely:

In [ ]:
K="kubectl --context kind-mesh1"
arctl delete deployment prtriage 2>/dev/null || true
arctl delete agent prtriage 2>/dev/null || true
arctl delete mcpserver github-mcp 2>/dev/null || true
arctl delete skill release-report 2>/dev/null || true
$K -n agentgateway-system delete enterpriseagentgatewaypolicy github-readonly --ignore-not-found
$K -n agentgateway-system delete enterpriseagentgatewaybackend github-mcp --ignore-not-found
$K -n agentgateway-system delete httproute github-mcp --ignore-not-found
$K -n agentgateway-system delete secret github-mcp-pat --ignore-not-found
echo "✓ Part 8 removed"